In [15]:
from faker import Faker
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta
from pathlib import Path
import sqlite3

In [5]:
#making the directory
fake = Faker()
Faker.seed(42)


In [12]:
def generate_customer(n=1000):
    customers = []
    for i in range(n):
        customer = {
            "customer_id": i + 1,
            "first_name": fake.first_name(),
            "last_name": fake.last_name(),
            "email":fake.email(),
            "phone_number": fake.phone_number(),
            'signup_date': fake.date_between(start_date='-2y', end_date='today'),
            'country': fake.country(),
            'customer_tier': random.choice(['Bronze', 'Silver', 'Gold', 'Platinum'])
        }
        customers.append(customer)
    
   
    df = pd.DataFrame(customers)

    return df

def generate_orders(customer_ids, n=5000):
    """Generate clean orders with NO quality issues"""
    orders = []
    
    for i in range(n):
        order_date = fake.date_time_between(start_date='-1y', end_date='now')
        
        order = {
            'order_id': i + 1,
            'customer_id': random.choice(customer_ids),
            'order_date': order_date,
            'order_total': round(random.uniform(10, 500), 2),
            'status': random.choice(['completed', 'pending', 'cancelled', 'shipped']),
            'payment_method': random.choice(['credit_card', 'paypal', 'bank_transfer', 'crypto'])
        }
        orders.append(order)
    
    df = pd.DataFrame(orders)    
    return df

def generate_products(n=100):
    """Generate clean products with NO quality issues"""
    products = []
    
    categories = ['Electronics', 'Clothing', 'Food & Beverage', 'Books', 'Toys', 'Home & Garden']
    
    for i in range(n):
        product = {
            'product_id': i + 1,
            'product_name': fake.catch_phrase(),
            'price': round(random.uniform(5, 200), 2),
            'category': random.choice(categories),
            'stock_quantity': random.randint(0, 500),
            'supplier': fake.company()
        }
        products.append(product)
    
    df = pd.DataFrame(products)    
    return df

def generate_order_items(order_ids, product_ids, n=8000):
    """Generate clean order items with NO quality issues"""
    order_items = []
    
    for i in range(n):
        order_item = {
            'order_item_id': i + 1,
            'order_id': random.choice(order_ids),
            'product_id': random.choice(product_ids),
            'quantity': random.randint(1, 5),
            'unit_price': round(random.uniform(5, 200), 2)
        }
        order_items.append(order_item)
    
    df = pd.DataFrame(order_items)
    
    return df



customer_data = generate_customer()
product_data = generate_products()
order_data = generate_orders(customer_data['customer_id'].tolist())
order_item_data = generate_order_items(order_data['order_id'].tolist(), list(range(1, 101)))




In [13]:
#injecting errors to test data quality tools:

def inject_errors(
    customers_df,
    orders_df,
    products_df,
    # Customer errors
    null_emails=True,
    null_email_pct=0.03,
    duplicate_customers=True,
    duplicate_count=20,
    # Order errors
    orphaned_orders=True,
    orphaned_pct=0.02,
    null_order_totals=True,
    null_total_pct=0.01,
    volume_anomaly=True,
    anomaly_date='2024-03-15',
    anomaly_reduction=0.9,
    # Product errors
    price_outliers=True,
    outlier_count=5,
    outlier_values=None
):
    """
    Inject data quality errors into all tables
    
    Parameters:
    -----------
    customers_df, orders_df, products_df : pandas.DataFrame
        Clean dataframes
    
    CUSTOMER ERRORS:
    null_emails : bool - Inject null email addresses
    null_email_pct : float - Percentage of emails to null (default 3%)
    duplicate_customers : bool - Inject duplicate customer records
    duplicate_count : int - Number of duplicates to add (default 20)
    
    ORDER ERRORS:
    orphaned_orders : bool - Inject orphaned records (invalid foreign keys)
    orphaned_pct : float - Percentage of orders to orphan (default 2%)
    null_order_totals : bool - Inject null order totals
    null_total_pct : float - Percentage of totals to null (default 1%)
    volume_anomaly : bool - Create volume anomaly on specific date
    anomaly_date : str - Date for anomaly (default '2024-03-15')
    anomaly_reduction : float - Fraction of orders to remove (default 0.9)
    
    PRODUCT ERRORS:
    price_outliers : bool - Inject price outliers
    outlier_count : int - Number of outliers (default 5)
    outlier_values : list - Specific outlier prices (optional)
    
    Returns:
    --------
    tuple : (customers_df, orders_df, products_df) with errors injected
    """
    
    print("\n" + "=" * 60)
    print("INJECTING DATA QUALITY ERRORS")
    print("=" * 60 + "\n")
    
    # Copy dataframes to avoid modifying originals
    customers = customers_df.copy()
    orders = orders_df.copy()
    products = products_df.copy()
    
    error_summary = []
    
    # ========== CUSTOMER ERRORS ==========
    print("CUSTOMERS TABLE")
    
    if null_emails:
        null_indices = customers.sample(frac=null_email_pct).index
        customers.loc[null_indices, 'email'] = None
        null_count = customers['email'].isnull().sum()
        print(f"{null_count} null emails ({null_count/len(customers)*100:.1f}%)")
        error_summary.append(f"{null_count} null emails")
    
    if duplicate_customers:
        duplicates_to_add = []
        for i in range(duplicate_count):
            source_idx = random.randint(0, len(customers_df) - 1)
            duplicate = customers.iloc[source_idx].copy()
            duplicate['customer_id'] = customers['customer_id'].max() + i + 1
            duplicate['signup_date'] = fake.date_between(start_date='-1y', end_date='today')
            duplicates_to_add.append(duplicate)
        
        customers = pd.concat([customers, pd.DataFrame(duplicates_to_add)], ignore_index=True)
        print(f"{duplicate_count} duplicate records")
        error_summary.append(f"{duplicate_count} duplicates")
    
    print(f"Final count: {len(customers)}\n")
    
    # ========== ORDER ERRORS ==========
    print("📋 ORDERS TABLE")
    
    if orphaned_orders:
        orphaned_indices = orders.sample(frac=orphaned_pct).index
        orders.loc[orphaned_indices, 'customer_id'] = 99999
        orphaned_count = (orders['customer_id'] == 99999).sum()
        print(f"{orphaned_count} orphaned orders ({orphaned_count/len(orders)*100:.1f}%)")
        error_summary.append(f"{orphaned_count} orphaned orders")
    
    if null_order_totals:
        null_indices = orders.sample(frac=null_total_pct).index
        orders.loc[null_indices, 'order_total'] = None
        null_count = orders['order_total'].isnull().sum()
        print(f"{null_count} null order totals ({null_count/len(orders)*100:.1f}%)")
        error_summary.append(f"{null_count} null totals")
    
    if volume_anomaly:
        orders['order_date'] = pd.to_datetime(orders['order_date'])
        anomaly_date_parsed = pd.to_datetime(anomaly_date)
        anomaly_mask = orders['order_date'].dt.date == anomaly_date_parsed.date()
        orders_on_date = anomaly_mask.sum()
        
        if orders_on_date > 0:
            orders_to_remove = int(orders_on_date * anomaly_reduction)
            anomaly_indices = orders[anomaly_mask].sample(n=orders_to_remove).index
            orders = orders.drop(anomaly_indices)
            
            remaining = (orders['order_date'].dt.date == anomaly_date_parsed.date()).sum()
            avg_daily = orders.groupby(orders['order_date'].dt.date).size().mean()
            
            print(f"Volume anomaly on {anomaly_date}: {remaining} orders (avg: {avg_daily:.0f})")
            error_summary.append(f"Volume anomaly on {anomaly_date}")
    
    print(f"Final count: {len(orders)}\n")
    
    # ========== PRODUCT ERRORS ==========
    print("PRODUCTS TABLE")
    
    if price_outliers:
        if outlier_values is None:
            outlier_values = [0.01, -50.00, 99999.99, 0.00, 150000.00]
        
        outliers_to_add = []
        for i in range(min(outlier_count, len(outlier_values))):
            outlier = {
                'product_id': products['product_id'].max() + i + 1,
                'product_name': fake.catch_phrase(),
                'price': outlier_values[i],
                'category': random.choice(products['category'].unique()),
                'stock_quantity': random.randint(0, 500),
                'supplier': fake.company()
            }
            outliers_to_add.append(outlier)
        
        products = pd.concat([products, pd.DataFrame(outliers_to_add)], ignore_index=True)
        outlier_count_actual = ((products['price'] < 1) | (products['price'] > 1000)).sum()
        print(f"{len(outliers_to_add)} price outliers (total <$1 or >$1000: {outlier_count_actual})")
        error_summary.append(f"{outlier_count_actual} price outliers")
    
    print(f"Final count: {len(products)}\n")
    
    # Summary
    print("=" * 60)
    print("ERROR INJECTION COMPLETE")
    print("=" * 60)
    print(f"Injected {len(error_summary)} types of errors:")
    for error in error_summary:
        print(f"   • {error}")
    print()
    
    return customers, orders, products

In [ ]:
def create_database(inject_data_errors=False, **error_params):
    """
    Generate database with optional error injection
    
    Parameters:
    -----------
    inject_data_errors : bool
        Whether to inject any errors (default True)
    **error_params : dict
        Any parameters to pass to inject_errors() function
        
    Examples:
    ---------
    # Default errors
    create_database()
    
    # Clean data
    create_database(inject_data_errors=False)
    
    # Custom errors
    create_database(
        null_email_pct=0.10,
        duplicate_count=50,
        orphaned_pct=0.05
    )
    """
    
    print("=" * 60)
    print("GENERATING DATA QUALITY FRAMEWORK DEMO DATABASE")
    print("=" * 60 + "\n")
    
    # Generate clean data
    customers_df = generate_customer(1000)
    orders_df = generate_orders(customers_df['customer_id'].tolist(), 5000)
    products_df = generate_products(100)
    order_items_df = generate_order_items(
        orders_df['order_id'].tolist(),
        products_df['product_id'].tolist(),
        8000
    )
    
    # Inject errors if requested
    if inject_data_errors:
        customers_df, orders_df, products_df = inject_errors(
            customers_df,
            orders_df,
            products_df,
            **error_params
        )
    
    # Create SQLite database
    db_path = '../data/raw/s`ample_ecommerce.db'
    print("=" * 60)
    print(f"SAVING TO DATABASE: {db_path}")
    print("=" * 60 + "\n")
    
    conn = sqlite3.connect(db_path)


     # Save tables
    print("Writing tables...")
    customers_df.to_sql('customers', conn, if_exists='replace', index=False)
    orders_df.to_sql('orders', conn, if_exists='replace', index=False)
    products_df.to_sql('products', conn, if_exists='replace', index=False)
    order_items_df.to_sql('order_items', conn, if_exists='replace', index=False)
    conn.commit()
    conn.close()
    
    # Final summary
    print("\n" + "=" * 60)
    print("DATABASE CREATED SUCCESSFULLY")
    print("=" * 60)
    print(f"\nLocation: {db_path}\n")
    print("Final Table Counts:")
    print(f"   • {len(customers_df):,} customers")
    print(f"   • {len(orders_df):,} orders")
    print(f"   • {len(products_df):,} products")
    print(f"   • {len(order_items_df):,} order items")
    
    if inject_data_errors:
        print("\nData Quality Issues Present:")
        
        null_emails = customers_df['email'].isnull().sum()
        if null_emails > 0:
            print(f"{null_emails} null emails ({null_emails/len(customers_df)*100:.1f}%)")
        
        duplicate_emails = customers_df[customers_df['email'].notna()]['email'].duplicated().sum()
        if duplicate_emails > 0:
            print(f"{duplicate_emails} duplicate customers")
        
        orphaned = (orders_df['customer_id'] == 99999).sum()
        if orphaned > 0:
            print(f"{orphaned} orphaned orders")
        
        null_totals = orders_df['order_total'].isnull().sum()
        if null_totals > 0:
            print(f"{null_totals} null order totals")
        
        outliers = ((products_df['price'] < 1) | (products_df['price'] > 1000)).sum()
        if outliers > 0:
            print(f"{outliers} price outliers")
    else:
        print("\nCLEAN DATA (no errors injected)")
    
    print("\nNext Steps:")
    print("   1. Open in DB Browser: data/raw/sample_ecommerce.db")
    print("   2. Run: python scripts/verify_data.py")
    print("   3. Start building your test engine!")
    print("\n" + "=" * 60 + "\n")


In [18]:
if __name__ == "__main__":
    create_database()

GENERATING DATA QUALITY FRAMEWORK DEMO DATABASE

SAVING TO DATABASE: ../data/raw/sample_ecommerce.db

Writing tables...

DATABASE CREATED SUCCESSFULLY

Location: ../data/raw/sample_ecommerce.db

Final Table Counts:
   • 1,000 customers
   • 5,000 orders
   • 100 products
   • 8,000 order items
